# 03 — Dataset Merge (Synthetic Linkage)
## SBDR Phase A, Step A3
Merge all 5 processed datasets into one multi-modal record per customer.
UCI Credit Card (30K customers) is our anchor — everything maps onto these IDs.

In [1]:
import pandas as pd
import numpy as np

# Load all processed datasets
uci = pd.read_csv("../data/processed/uci_credit_processed.csv")
lc = pd.read_csv("../data/processed/lending_club_processed.csv")
sparkov = pd.read_csv("../data/processed/sparkov_processed.csv")
phrasebank = pd.read_csv("../data/processed/financial_phrasebank_processed.csv")
mental = pd.read_csv("../data/processed/mental_health_processed.csv")

print("=== All Datasets Loaded ===")
for name, df in [("UCI", uci), ("Lending Club", lc), ("Sparkov", sparkov), 
                  ("PhraseBank", phrasebank), ("Mental Health", mental)]:
    print(f"{name:15s} → {df.shape[0]:>10,} rows × {df.shape[1]} cols")

=== All Datasets Loaded ===
UCI             →     30,000 rows × 32 cols
Lending Club    →  2,257,919 rows × 19 cols
Sparkov         →  1,852,394 rows × 9 cols
PhraseBank      →      2,264 rows × 2 cols
Mental Health   →     38,175 rows × 2 cols


## Step 1: Create Customer IDs & Distress Levels
Each UCI customer gets a unique ID. Their payment history (PAY_0 to PAY_6) 
determines a "distress_level" which controls how we assign sentiment text and 
Lending Club profiles to them.

In [2]:
# Assign unique customer IDs
uci['customer_id'] = [f"SBDR_{i:05d}" for i in range(len(uci))]

# Compute distress level from payment history
# PAY columns: -2=no usage, -1=paid on time, 0=revolving, 1+=months late
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

# Average payment delay across 6 months (higher = worse)
uci['avg_pay_delay'] = uci[pay_cols].mean(axis=1)

# Distress level: 4 bins matching our 4 SBDR tiers
uci['distress_level'] = pd.cut(
    uci['avg_pay_delay'],
    bins=[-np.inf, -0.5, 0.0, 1.0, np.inf],
    labels=['low', 'moderate', 'high', 'severe']
)

print("=== Distress Level Distribution ===")
print(uci['distress_level'].value_counts().sort_index())
print(f"\navg_pay_delay range: {uci['avg_pay_delay'].min():.2f} to {uci['avg_pay_delay'].max():.2f}")

=== Distress Level Distribution ===
distress_level
low         10253
moderate    12614
high         4500
severe       2633
Name: count, dtype: int64

avg_pay_delay range: -2.00 to 6.00


## Step 2: Enrich with Lending Club Profiles
We compute aggregate stats from the full 2.26M Lending Club loans, 
grouped by risk tier. Then map those stats onto UCI customers 
based on their distress level.

In [3]:
# Map Lending Club sbdr_tier to our distress levels
tier_to_distress = {
    'No Action': 'low',
    'Tier 1 - Reminder': 'low',
    'Tier 2 - Assistance': 'moderate',
    'Tier 3 - Restructure': 'high',
    'Tier 4 - Legal': 'severe'
}
lc['distress_level'] = lc['sbdr_tier'].map(tier_to_distress)

# Drop rows that didn't map (e.g. "Current" loans with no tier)
lc_mapped = lc.dropna(subset=['distress_level'])
print(f"Lending Club rows with distress mapping: {len(lc_mapped):,} / {len(lc):,}")

# Compute aggregate profiles per distress level
# These are the numeric columns we want stats for
lc_numeric = ['loan_amnt', 'funded_amnt', 'annual_inc', 'dti', 
              'revol_util', 'delinq_2yrs', 'inq_last_6mths', 
              'open_acc', 'pub_rec', 'total_acc', 'installment', 'int_rate']

lc_profiles = lc_mapped.groupby('distress_level')[lc_numeric].agg(['mean', 'std']).reset_index()

# Flatten column names: "loan_amnt_mean", "loan_amnt_std", etc.
lc_profiles.columns = ['distress_level'] + [f"lc_{col}_{stat}" for col, stat in lc_profiles.columns[1:]]

print(f"\nLC profile shape: {lc_profiles.shape}")
print(f"Distress levels in profiles: {lc_profiles['distress_level'].tolist()}")

Lending Club rows with distress mapping: 1,955,068 / 2,257,919

LC profile shape: (1, 25)
Distress levels in profiles: ['low']


In [4]:
print("=== Lending Club sbdr_tier distribution ===")
print(lc['sbdr_tier'].value_counts())

=== Lending Club sbdr_tier distribution ===
sbdr_tier
No Action    1955068
Tier 4        268599
Tier 3         21467
Tier 1          8436
Tier 2          4349
Name: count, dtype: int64


---
Problem — almost everything mapped to "low."

---

In [5]:
# Correct mapping with actual tier labels
tier_to_distress = {
    'No Action': 'low',
    'Tier 1': 'low',
    'Tier 2': 'moderate',
    'Tier 3': 'high',
    'Tier 4': 'severe'
}
lc['distress_level'] = lc['sbdr_tier'].map(tier_to_distress)

lc_mapped = lc.dropna(subset=['distress_level'])
print(f"Lending Club rows with distress mapping: {len(lc_mapped):,} / {len(lc):,}")
print(f"\n=== LC Distress Distribution ===")
print(lc_mapped['distress_level'].value_counts().sort_index())

# Recompute profiles
lc_numeric = ['loan_amnt', 'funded_amnt', 'annual_inc', 'dti', 
              'revol_util', 'delinq_2yrs', 'inq_last_6mths', 
              'open_acc', 'pub_rec', 'total_acc', 'installment', 'int_rate']

lc_profiles = lc_mapped.groupby('distress_level')[lc_numeric].agg(['mean', 'std']).reset_index()
lc_profiles.columns = ['distress_level'] + [f"lc_{col}_{stat}" for col, stat in lc_profiles.columns[1:]]

print(f"\nLC profile shape: {lc_profiles.shape}")
print(f"Distress levels in profiles: {lc_profiles['distress_level'].tolist()}")

Lending Club rows with distress mapping: 2,257,919 / 2,257,919

=== LC Distress Distribution ===
distress_level
high          21467
low         1963504
moderate       4349
severe       268599
Name: count, dtype: int64

LC profile shape: (4, 25)
Distress levels in profiles: ['high', 'low', 'moderate', 'severe']


---
All 4 profiles confirmed, full 2.26M rows contributing. The "moderate" and "high" buckets are smaller but still have thousands of rows each — more than enough for reliable aggregate stats.

---

In [6]:
# Merge LC aggregate profiles onto UCI customers by distress level
uci_merged = uci.merge(lc_profiles, on='distress_level', how='left')

# Add realistic noise so not every "low" customer has identical LC stats
# Without noise, all 10,253 "low" customers would have the exact same values
np.random.seed(42)
lc_feature_cols = [c for c in uci_merged.columns if c.startswith('lc_') and c.endswith('_mean')]

for col in lc_feature_cols:
    std_col = col.replace('_mean', '_std')
    if std_col in uci_merged.columns:
        noise = np.random.normal(0, 1, len(uci_merged)) * uci_merged[std_col] * 0.3
        uci_merged[col] = uci_merged[col] + noise
        # Floor at 0 for columns that can't be negative
        if any(keyword in col for keyword in ['amnt', 'inc', 'acc', 'installment']):
            uci_merged[col] = uci_merged[col].clip(lower=0)

# Drop the std columns — they were only needed for noise generation
std_cols = [c for c in uci_merged.columns if c.startswith('lc_') and c.endswith('_std')]
uci_merged.drop(columns=std_cols, inplace=True)

print(f"Shape after LC merge: {uci_merged.shape}")
print(f"New LC columns: {[c for c in uci_merged.columns if c.startswith('lc_')]}")
print(f"Null check: {uci_merged[[c for c in uci_merged.columns if c.startswith('lc_')]].isnull().sum().sum()}")

Shape after LC merge: (30000, 47)
New LC columns: ['lc_loan_amnt_mean', 'lc_funded_amnt_mean', 'lc_annual_inc_mean', 'lc_dti_mean', 'lc_revol_util_mean', 'lc_delinq_2yrs_mean', 'lc_inq_last_6mths_mean', 'lc_open_acc_mean', 'lc_pub_rec_mean', 'lc_total_acc_mean', 'lc_installment_mean', 'lc_int_rate_mean']
Null check: 0


## Step 3: Assign Sparkov Spending Profiles
Sparkov has 999 customers with full transaction histories. We compute 
monthly spending profiles per customer (total spend, volatility, top category), 
then assign each Sparkov profile to ~30 UCI customers based on similarity.

In [7]:
# Parse transaction dates
sparkov['transaction_date'] = pd.to_datetime(sparkov['transaction_date'])

# Compute per-customer spending profiles from full transaction history
sparkov_profiles = sparkov.groupby('customer_id').agg(
    total_spend=('amount', 'sum'),
    avg_monthly_spend=('amount', 'mean'),
    spend_volatility=('amount', 'std'),
    num_transactions=('amount', 'count'),
    top_category=('category', lambda x: x.mode()[0]),
    fraud_rate=('is_fraud', 'mean')
).reset_index()

# Rank customers by spending level (quartiles)
sparkov_profiles['spend_quartile'] = pd.qcut(
    sparkov_profiles['total_spend'], 
    q=4, labels=['q1_low', 'q2_mid', 'q3_high', 'q4_heavy']
)

print(f"Sparkov profiles: {sparkov_profiles.shape}")
print(f"\n=== Spend Quartile Distribution ===")
print(sparkov_profiles['spend_quartile'].value_counts().sort_index())
print(f"\n=== Top 5 Categories ===")
print(sparkov_profiles['top_category'].value_counts().head())

Sparkov profiles: (999, 8)

=== Spend Quartile Distribution ===
spend_quartile
q1_low      250
q2_mid      250
q3_high     249
q4_heavy    250
Name: count, dtype: int64

=== Top 5 Categories ===
top_category
gas_transport    475
grocery_pos      179
shopping_pos     136
home             118
shopping_net      40
Name: count, dtype: int64
